amazon Insertion

In [ ]:
import pandas as pd
import os
import random
import torch

# Set random seed
random.seed(2024)

# ==================== Configuration Parameters ====================
input_path = 'C:/Users/THINK BOOK-16/Desktop/beauty/'  # Directory containing inter.csv for toy dataset
output_path = 'C:/Users/THINK BOOK-16/Desktop/beauty-processed-noise-10/'
dataset_name = 'beauty'

user_threshold = 5
item_threshold = 5
max_seq_len = 50

# ==================== Generate noise version with only 10% ====================
NOISE_RATIO = 0.10  # Add 10% noise

# ==================== Helper Functions ====================
def add_noise_insert_random(dataset, noise_ratio=0.05):
    """
    Add noise: randomly insert new interaction records
    Requirements:
    1. Select from existing users and items
    2. Rating and timestamp are selected and randomly combined from existing sequences
    3. Domain remains unchanged
    4. Precisely control noise ratio to 5%
    """
    print(f"\n  🔧 Starting to add noise (inserting random combined interactions)...")
    
    # Get dataset info
    original_size = len(dataset)
    target_new_interactions = int(original_size * noise_ratio)
    
    print(f"  Original interactions: {original_size}")
    print(f"  Target new interactions: {target_new_interactions} (expected noise ratio: {noise_ratio*100:.1f}%)")
    
    # Copy dataset
    noisy_dataset = dataset.copy()
    
    # Get all possible users and items
    all_users = dataset['user_id'].unique().tolist()
    all_items = dataset['item_id'].unique().tolist()
    
    # Get rating and timestamp range and distribution
    if 'rating' in dataset.columns:
        all_ratings = dataset['rating'].unique().tolist()
        rating_column_exists = True
    else:
        rating_column_exists = False
        # If no rating column, create a default value
        all_ratings = [1]  # default rating
    
    # Get timestamp range
    min_timestamp = dataset['timestamp'].min()
    max_timestamp = dataset['timestamp'].max()
    
    print(f"  Available users: {len(all_users)}")
    print(f"  Available items: {len(all_items)}")
    if rating_column_exists:
        print(f"  Available rating values: {len(all_ratings)}")
    print(f"  Timestamp range: {min_timestamp} ~ {max_timestamp}")
    
    # Generate new noise interactions
    new_interactions = []
    added_count = 0
    
    while added_count < target_new_interactions:
        # Randomly select user and item
        user = random.choice(all_users)
        item = random.choice(all_items)
        
        # Randomly select rating (if exists)
        if rating_column_exists:
            rating = random.choice(all_ratings)
        else:
            rating = 1
        
        # Randomly generate timestamp (within existing range)
        timestamp = random.randint(min_timestamp, max_timestamp)
        
        # Create new interaction record - fully maintain original column structure
        new_interaction = {
            'user_id': int(user),  # Ensure Python int type
            'item_id': int(item),  # Ensure Python int type
            'timestamp': int(timestamp)
        }
        
        # Add domain column (if exists)
        if 'domain' in dataset.columns:
            # Get domain value from original data (consistent with user)
            domain_val = dataset[dataset['user_id'] == user]['domain'].iloc[0] if len(dataset[dataset['user_id'] == user]) > 0 else 0
            new_interaction['domain'] = int(domain_val)
        
        if rating_column_exists:
            new_interaction['rating'] = rating
        
        # Add to list
        new_interactions.append(new_interaction)
        added_count += 1
        
        # Show progress
        if added_count % 1000 == 0 or added_count == target_new_interactions:
            progress = added_count / target_new_interactions * 100
            print(f"    Progress: {added_count}/{target_new_interactions} ({progress:.1f}%)")
    
    # Convert new interactions to DataFrame
    new_interactions_df = pd.DataFrame(new_interactions)
    
    # Merge with original dataset
    noisy_dataset = pd.concat([noisy_dataset, new_interactions_df], ignore_index=True)
    
    # Sort by user and time
    noisy_dataset = noisy_dataset.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)
    
    # Calculate actual noise ratio
    total_interactions = len(noisy_dataset)
    actual_noise_ratio = (total_interactions - original_size) / original_size
    
    print(f"\n  📊 Noise addition completed:")
    print(f"    Original interactions: {original_size}")
    print(f"    Added interactions: {added_count}")
    print(f"    Total interactions: {total_interactions}")
    print(f"    Actual noise ratio: {actual_noise_ratio*100:.2f}%")
    
    return noisy_dataset, added_count, original_size

def truncate_or_pad(seq):
    """Truncate or pad sequence to fixed length (maintain original logic)"""
    cur_seq_len = len(seq)
    if cur_seq_len > max_seq_len:
        return seq[-max_seq_len:], max_seq_len
    else:
        PAD = 0
        return seq + [PAD] * (max_seq_len - cur_seq_len), cur_seq_len

def ensure_python_types(data):
    """Ensure all data is Python native types instead of numpy types"""
    if isinstance(data, np.integer):
        return int(data)
    elif isinstance(data, np.floating):
        return float(data)
    elif isinstance(data, np.ndarray):
        return data.tolist()
    elif isinstance(data, list):
        return [ensure_python_types(item) for item in data]
    elif isinstance(data, dict):
        return {key: ensure_python_types(value) for key, value in data.items()}
    else:
        return data

# ==================== Main Process ====================

print("="*60)
print("Starting Toy Dataset Processing (10% noise insertion, random combination)")
print("="*60)

# 1. Load inter.csv file
print("\n1. Loading inter.csv file...")
inter_file = os.path.join(input_path, 'inter.csv')

if not os.path.exists(inter_file):
    print(f"Error: Cannot find file {inter_file}")
    exit(1)

# Read inter.csv file
dataset = pd.read_csv(inter_file)

print(f"Original data statistics:")
print(f"  Data shape: {dataset.shape}")
print(f"  Columns: {dataset.columns.tolist()}")
print(f"  Number of users: {dataset['user_id'].nunique()}")
print(f"  Number of items: {dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(dataset)}")

# 2. Filter dataset (based on interaction frequency)
print("\n2. Filtering dataset...")
filtered_dataset = dataset.copy()
while True:
    ori_len = len(filtered_dataset)
    
    # Filter users with less than user_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['user_id'].map(filtered_dataset['user_id'].value_counts()) >= user_threshold]
    # Filter items with less than item_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['item_id'].map(filtered_dataset['item_id'].value_counts()) >= item_threshold]
    
    if len(filtered_dataset) == ori_len:
        break

print(f"\nFiltered data statistics:")
print(f"  Number of users: {filtered_dataset['user_id'].nunique()}")
print(f"  Number of items: {filtered_dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(filtered_dataset)}")
print(f"  Item interaction range: {filtered_dataset['item_id'].value_counts().min()} ~ {filtered_dataset['item_id'].value_counts().max()}")

# 3. Remap IDs (consistent with original code)
print("\n3. Remapping IDs...")
all_user = filtered_dataset.user_id
all_item = filtered_dataset.item_id

user_id, user_token = pd.factorize(all_user)
item_id, item_token = pd.factorize(all_item)

num_users = len(user_token) + 1  # 0 id is for PAD
num_items = len(item_token) + 1  # 0 id is for PAD

user_mapping_dict = {_: idx + 1 for idx, _ in enumerate(user_token)}  # 0 id is for PAD
item_mapping_dict = {_: idx + 1 for idx, _ in enumerate(item_token)}  # 0 id is for PAD

print(f"User mapping: {user_token.shape}")
print(f"Item mapping: {item_token.shape}")

filtered_dataset['user_id'] = filtered_dataset['user_id'].apply(lambda x: user_mapping_dict[x])
filtered_dataset['item_id'] = filtered_dataset['item_id'].apply(lambda x: item_mapping_dict[x])

# Check if domain column exists, if not add one (following original code logic)
if 'domain' not in filtered_dataset.columns:
    print("Note: No domain column in data, adding default domain=0")
    filtered_dataset['domain'] = 0

# Ensure domain is integer type
filtered_dataset['domain'] = filtered_dataset['domain'].astype(int)

# Save clean item list and counts
clean_items = sorted(filtered_dataset['item_id'].unique())
clean_item_counts = filtered_dataset['item_id'].value_counts().to_dict()
print(f"Clean item count: {len(clean_items)}")
print(f"Clean item ID range: {clean_items[0]} ~ {clean_items[-1]}")
print(f"Clean minimum interaction count: {min(clean_item_counts.values())}")

# 4. Create output directory
os.makedirs(output_path, exist_ok=True)

# 5. Process clean data
print(f"\n{'='*40}")
print(f"Processing clean data...")
print(f"{'='*40}")

clean_dataset = filtered_dataset.copy()
clean_type = 'clean'

# Create output directory
clean_dir = os.path.join(output_path, dataset_name, clean_type)
os.makedirs(clean_dir, exist_ok=True)

# Save inter.csv
csv_path = os.path.join(clean_dir, 'inter.csv')
clean_dataset.to_csv(csv_path, sep=',', index=None)
print(f"  Saving {clean_type} interaction data to: {csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
clean_dataset_sorted = clean_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
def to_list(x):
    return list(x)[:-2]  # Remove last 2 interactions, maintain original logic

user_group_for_seq2pat = clean_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Filter out empty lists
seq2pat_data = [seq for seq in user_group_for_seq2pat.tolist() if len(seq) > 0]

# Ensure data types are Python native types
seq2pat_data = ensure_python_types(seq2pat_data)

# Save seq2pat_data.pth
seq2pat_path = os.path.join(clean_dir, 'seq2pat_data.pth')
torch.save(seq2pat_data, seq2pat_path)
print(f"  Generated {clean_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data)}")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group = clean_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train, val, test = [], [], []

PAD = 0

# Process each user
for user_id, user_seq in list(zip(user_group.index, user_group.tolist())):
    # Ensure user_id is Python int type
    user_id = int(user_id)
    
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # Ensure item_id in user_seq is Python int type
    user_seq = [int(item) for item in user_seq]
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    
    # Domain handling: following original code logic, domain_id is [0] * max_seq_len
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    test.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    val.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    # Maintain original logic
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    train.append([user_id, history, target_data, seq_len, label, domain_id])

# Ensure all data are Python native types
train = ensure_python_types(train)
val = ensure_python_types(val)
test = ensure_python_types(test)

# Save sequence data
torch.save(train, os.path.join(clean_dir, 'train.pth'))
torch.save(val, os.path.join(clean_dir, 'val.pth'))
torch.save(test, os.path.join(clean_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train)}, validation={len(val)}, test={len(test)}")

# 6. Generate 10% noise data (random insertion)
print(f"\n{'='*40}")
print(f"Processing 10% noise data (random insertion of new interactions)...")
print(f"{'='*40}")

# Generate noise data
noisy_dataset, added_count, original_size = add_noise_insert_random(
    filtered_dataset.copy(), 
    noise_ratio=NOISE_RATIO
)

noisy_type = f'noise_{int(NOISE_RATIO*100)}'

# Create output directory
noisy_dir = os.path.join(output_path, dataset_name, noisy_type)
os.makedirs(noisy_dir, exist_ok=True)

# Save inter.csv
noisy_csv_path = os.path.join(noisy_dir, 'inter.csv')
noisy_dataset.to_csv(noisy_csv_path, sep=',', index=None)
print(f"  Saving {noisy_type} interaction data to: {noisy_csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
noisy_dataset_sorted = noisy_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
user_group_for_seq2pat_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Filter out empty lists
seq2pat_data_noisy = [seq for seq in user_group_for_seq2pat_noisy.tolist() if len(seq) > 0]

# Ensure data types are Python native types
seq2pat_data_noisy = ensure_python_types(seq2pat_data_noisy)

# Save seq2pat_data.pth
seq2pat_path_noisy = os.path.join(noisy_dir, 'seq2pat_data.pth')
torch.save(seq2pat_data_noisy, seq2pat_path_noisy)
print(f"  Generated {noisy_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data_noisy)}")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train_noisy, val_noisy, test_noisy = [], [], []

# Process each user
for user_id, user_seq in list(zip(user_group_noisy.index, user_group_noisy.tolist())):
    # Ensure user_id is Python int type
    user_id = int(user_id)
    
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # Ensure item_id in user_seq is Python int type
    user_seq = [int(item) for item in user_seq]
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    
    # Domain handling: following original code logic, domain_id is [0] * max_seq_len
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    test_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    val_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    train_noisy.append([user_id, history, target_data, seq_len, label, domain_id])

# Ensure all data are Python native types
train_noisy = ensure_python_types(train_noisy)
val_noisy = ensure_python_types(val_noisy)
test_noisy = ensure_python_types(test_noisy)

# Save sequence data
torch.save(train_noisy, os.path.join(noisy_dir, 'train.pth'))
torch.save(val_noisy, os.path.join(noisy_dir, 'val.pth'))
torch.save(test_noisy, os.path.join(noisy_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train_noisy)}, validation={len(val_noisy)}, test={len(test_noisy)}")

print("\n" + "="*60)
print("Data processing completed!")
print("="*60)

# 7. Final verification
print(f"\n{'='*40}")
print("Final verification results:")
print(f"{'='*40}")

# Verify data integrity
df_clean = pd.read_csv(csv_path)
df_noisy = pd.read_csv(noisy_csv_path)

print(f"Clean data: {len(df_clean)} rows, {df_clean['user_id'].nunique()} users")
print(f"Noisy data: {len(df_noisy)} rows, {df_noisy['user_id'].nunique()} users")

# Calculate actual noise ratio
actual_noise_ratio = (len(df_noisy) - len(df_clean)) / len(df_clean)
print(f"\nNoise ratio statistics:")
print(f"  Target noise ratio: {NOISE_RATIO*100:.1f}%")
print(f"  Actual noise ratio: {actual_noise_ratio*100:.2f}%")
print(f"  Absolute error: {abs(actual_noise_ratio - NOISE_RATIO)*100:.3f}%")

print(f"\nOutput directory structure:")
print(f"{output_path}/")
print(f"└── {dataset_name}/")
print(f"    ├── clean/")
print(f"    │   ├── inter.csv")
print(f"    │   ├── seq2pat_data.pth")
print(f"    │   ├── train.pth")
print(f"    │   ├── val.pth")
print(f"    │   └── test.pth")
print(f"    └── noise_5/")
print(f"        ├── inter.csv")
print(f"        ├── seq2pat_data.pth")
print(f"        ├── train.pth")
print(f"        ├── val.pth")
print(f"        └── test.pth")

print(f"\nKey guarantees:")
print(f"1. All items have interaction count ≥ {item_threshold}")
print(f"2. Precisely controlled noise ratio ≈ 5%")
print(f"3. Random insertion of new interactions without modifying existing ones")
print(f"4. Domain handled according to original code logic (fixed to 0)")
print(f"5. All data are Python native types, avoiding numpy type issues")

print("\nToy dataset noise experiment ready!")

yelp Insertion

In [ ]:
import pandas as pd
import os
import random
import torch
import numpy as np

# Set random seed
random.seed(2024)

# ==================== Configuration Parameters ====================
input_path = 'C:/Users/THINK BOOK-16/Desktop/yelp/'  # Directory containing inter.csv for yelp dataset
output_path = 'C:/Users/THINK BOOK-16/Desktop/yelp-processed-noise-5/'
dataset_name = 'yelp'

user_threshold = 5
item_threshold = 5
max_seq_len = 50

# ==================== Generate only 5% noise version ====================
NOISE_RATIO = 0.05  # Add 5% noise

# ==================== Helper Functions ====================
def add_noise_insert_random(dataset, noise_ratio=0.05):
    """
    Add noise: randomly insert new interaction records
    Requirements:
    1. Select from existing users and items
    2. Rating and timestamp are selected and randomly combined from existing sequences
    3. Domain remains unchanged
    4. Precisely control noise ratio to 5%
    """
    print(f"\n  🔧 Starting to add noise (inserting random combined interactions)...")
    
    # Get dataset information
    original_size = len(dataset)
    target_new_interactions = int(original_size * noise_ratio)
    
    print(f"  Original interactions: {original_size}")
    print(f"  Target new interactions: {target_new_interactions} (expected noise ratio: {noise_ratio*100:.1f}%)")
    
    # Copy dataset
    noisy_dataset = dataset.copy()
    
    # Get all possible users and items
    all_users = dataset['user_id'].unique().tolist()
    all_items = dataset['item_id'].unique().tolist()
    
    # Get timestamp range
    min_timestamp = dataset['timestamp'].min()
    max_timestamp = dataset['timestamp'].max()
    
    print(f"  Available users: {len(all_users)}")
    print(f"  Available items: {len(all_items)}")
    print(f"  Timestamp range: {min_timestamp} ~ {max_timestamp}")
    
    # Generate new noise interactions
    new_interactions = []
    added_count = 0
    
    while added_count < target_new_interactions:
        # Randomly select user and item
        user = random.choice(all_users)
        item = random.choice(all_items)
        
        # Randomly generate timestamp (within existing range)
        timestamp = random.randint(min_timestamp, max_timestamp)
        
        # Create new interaction record - fully maintain original column structure
        new_interaction = {
            'user_id': int(user),  # Ensure Python int type
            'item_id': int(item),  # Ensure Python int type
            'timestamp': int(timestamp)
        }
        
        # Add domain column (if exists)
        if 'domain' in dataset.columns:
            # Get domain value from original data (consistent with user)
            domain_val = dataset[dataset['user_id'] == user]['domain'].iloc[0] if len(dataset[dataset['user_id'] == user]) > 0 else 0
            new_interaction['domain'] = int(domain_val)
        
        # Add to list
        new_interactions.append(new_interaction)
        added_count += 1
        
        # Show progress
        if added_count % 1000 == 0 or added_count == target_new_interactions:
            progress = added_count / target_new_interactions * 100
            print(f"    Progress: {added_count}/{target_new_interactions} ({progress:.1f}%)")
    
    # Convert new interactions to DataFrame
    new_interactions_df = pd.DataFrame(new_interactions)
    
    # Merge with original dataset
    noisy_dataset = pd.concat([noisy_dataset, new_interactions_df], ignore_index=True)
    
    # Sort by user and time
    noisy_dataset = noisy_dataset.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)
    
    # Calculate actual noise ratio
    total_interactions = len(noisy_dataset)
    actual_noise_ratio = (total_interactions - original_size) / original_size
    
    print(f"\n  Noise addition completed:")
    print(f"    Original interactions: {original_size}")
    print(f"    Added interactions: {added_count}")
    print(f"    Total interactions: {total_interactions}")
    print(f"    Actual noise ratio: {actual_noise_ratio*100:.2f}%")
    
    return noisy_dataset, added_count, original_size

def truncate_or_pad(seq):
    """Truncate or pad sequence to fixed length (maintain original logic)"""
    cur_seq_len = len(seq)
    if cur_seq_len > max_seq_len:
        return seq[-max_seq_len:], max_seq_len
    else:
        PAD = 0
        return seq + [PAD] * (max_seq_len - cur_seq_len), cur_seq_len

def ensure_python_types(data):
    """Ensure all data is Python native types instead of numpy types"""
    if isinstance(data, np.integer):
        return int(data)
    elif isinstance(data, np.floating):
        return float(data)
    elif isinstance(data, np.ndarray):
        return data.tolist()
    elif isinstance(data, list):
        return [ensure_python_types(item) for item in data]
    elif isinstance(data, dict):
        return {key: ensure_python_types(value) for key, value in data.items()}
    else:
        return data

# ==================== Main Process ====================

print("="*60)
print("Starting Yelp Dataset Processing (5% noise insertion, random combination)")
print("="*60)

# 1. Load inter.csv file
print("\n1. Loading inter.csv file...")
inter_file = os.path.join(input_path, 'inter.csv')

if not os.path.exists(inter_file):
    print(f"Error: Cannot find file {inter_file}")
    exit(1)

# Read inter.csv file
dataset = pd.read_csv(inter_file)

print(f"Original data statistics:")
print(f"  Data shape: {dataset.shape}")
print(f"  Column names: {dataset.columns.tolist()}")
print(f"  Number of users: {dataset['user_id'].nunique()}")
print(f"  Number of items: {dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(dataset)}")

# 2. Filter dataset (based on interaction frequency)
print("\n2. Filtering dataset...")
filtered_dataset = dataset.copy()
while True:
    ori_len = len(filtered_dataset)
    
    # Filter users with less than user_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['user_id'].map(filtered_dataset['user_id'].value_counts()) >= user_threshold]
    # Filter items with less than item_threshold interactions
    filtered_dataset = filtered_dataset[filtered_dataset['item_id'].map(filtered_dataset['item_id'].value_counts()) >= item_threshold]
    
    if len(filtered_dataset) == ori_len:
        break

print(f"\nFiltered data statistics:")
print(f"  Number of users: {filtered_dataset['user_id'].nunique()}")
print(f"  Number of items: {filtered_dataset['item_id'].nunique()}")
print(f"  Number of interactions: {len(filtered_dataset)}")
print(f"  Item interaction range: {filtered_dataset['item_id'].value_counts().min()} ~ {filtered_dataset['item_id'].value_counts().max()}")

# 3. Remap IDs (consistent with original code)
print("\n3. Remapping IDs...")
all_user = filtered_dataset.user_id
all_item = filtered_dataset.item_id

user_id, user_token = pd.factorize(all_user)
item_id, item_token = pd.factorize(all_item)

num_users = len(user_token) + 1  # 0 id is for PAD
num_items = len(item_token) + 1  # 0 id is for PAD

user_mapping_dict = {_: idx + 1 for idx, _ in enumerate(user_token)}  # 0 id is for PAD
item_mapping_dict = {_: idx + 1 for idx, _ in enumerate(item_token)}  # 0 id is for PAD

print(f"User mapping: {user_token.shape}")
print(f"Item mapping: {item_token.shape}")

filtered_dataset['user_id'] = filtered_dataset['user_id'].apply(lambda x: user_mapping_dict[x])
filtered_dataset['item_id'] = filtered_dataset['item_id'].apply(lambda x: item_mapping_dict[x])

# Check if domain column exists, if not add one (following original code logic)
if 'domain' not in filtered_dataset.columns:
    print("Note: No domain column in data, adding default domain=0")
    filtered_dataset['domain'] = 0

# Ensure domain is integer type
filtered_dataset['domain'] = filtered_dataset['domain'].astype(int)

# Save clean item list and counts
clean_items = sorted(filtered_dataset['item_id'].unique())
clean_item_counts = filtered_dataset['item_id'].value_counts().to_dict()
print(f"Clean item count: {len(clean_items)}")
print(f"Clean item ID range: {clean_items[0]} ~ {clean_items[-1]}")
print(f"Clean minimum interaction count: {min(clean_item_counts.values())}")

# 4. Create output directory
os.makedirs(output_path, exist_ok=True)

# 5. Process clean data
print(f"\n{'='*40}")
print(f"Processing clean data...")
print(f"{'='*40}")

clean_dataset = filtered_dataset.copy()
clean_type = 'clean'

# Create output directory
clean_dir = os.path.join(output_path, dataset_name, clean_type)
os.makedirs(clean_dir, exist_ok=True)

# Save inter.csv
csv_path = os.path.join(clean_dir, 'inter.csv')
clean_dataset.to_csv(csv_path, sep=',', index=None)
print(f"  Saving {clean_type} interaction data to: {csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
clean_dataset_sorted = clean_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
def to_list(x):
    return list(x)[:-2]  # Remove last 2 interactions, maintain original logic

user_group_for_seq2pat = clean_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Filter out empty lists
seq2pat_data = [seq for seq in user_group_for_seq2pat.tolist() if len(seq) > 0]

# Ensure data types are Python native types
seq2pat_data = ensure_python_types(seq2pat_data)

# Save seq2pat_data.pth
seq2pat_path = os.path.join(clean_dir, 'seq2pat_data.pth')
torch.save(seq2pat_data, seq2pat_path)
print(f"  Generated {clean_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data)}")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group = clean_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train, val, test = [], [], []

PAD = 0

# Process each user
for user_id, user_seq in list(zip(user_group.index, user_group.tolist())):
    # Ensure user_id is Python int type
    user_id = int(user_id)
    
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # Ensure item_id in user_seq is Python int type
    user_seq = [int(item) for item in user_seq]
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    
    # Domain handling: following original code logic, domain_id is [0] * max_seq_len
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    test.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    val.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    # Maintain original logic
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    train.append([user_id, history, target_data, seq_len, label, domain_id])

# Ensure all data are Python native types
train = ensure_python_types(train)
val = ensure_python_types(val)
test = ensure_python_types(test)

# Save sequence data
torch.save(train, os.path.join(clean_dir, 'train.pth'))
torch.save(val, os.path.join(clean_dir, 'val.pth'))
torch.save(test, os.path.join(clean_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train)}, validation={len(val)}, test={len(test)}")

# 6. Generate 5% noise data (random insertion)
print(f"\n{'='*40}")
print(f"Processing 5% noise data (random insertion of new interactions)...")
print(f"{'='*40}")

# Generate noise data
noisy_dataset, added_count, original_size = add_noise_insert_random(
    filtered_dataset.copy(), 
    noise_ratio=NOISE_RATIO
)

noisy_type = f'noise_{int(NOISE_RATIO*100)}'

# Create output directory
noisy_dir = os.path.join(output_path, dataset_name, noisy_type)
os.makedirs(noisy_dir, exist_ok=True)

# Save inter.csv
noisy_csv_path = os.path.join(noisy_dir, 'inter.csv')
noisy_dataset.to_csv(noisy_csv_path, sep=',', index=None)
print(f"  Saving {noisy_type} interaction data to: {noisy_csv_path}")

# Generate sequence data following original logic
print(f"  Generating sequence data...")

# Sort by user and time (original code logic)
noisy_dataset_sorted = noisy_dataset.sort_values(by=['user_id', 'timestamp'])

# ==================== Generate seq2pat_data.pth ====================
user_group_for_seq2pat_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(to_list)

# Filter out empty lists
seq2pat_data_noisy = [seq for seq in user_group_for_seq2pat_noisy.tolist() if len(seq) > 0]

# Ensure data types are Python native types
seq2pat_data_noisy = ensure_python_types(seq2pat_data_noisy)

# Save seq2pat_data.pth
seq2pat_path_noisy = os.path.join(noisy_dir, 'seq2pat_data.pth')
torch.save(seq2pat_data_noisy, seq2pat_path_noisy)
print(f"  Generated {noisy_type}/seq2pat_data.pth, valid sequences: {len(seq2pat_data_noisy)}")
# ==================== seq2pat_data generation ends ====================

# Continue generating train/val/test sequences
user_group_noisy = noisy_dataset_sorted.groupby('user_id')['item_id'].apply(list)

# Initialize lists
train_noisy, val_noisy, test_noisy = [], [], []

# Process each user
for user_id, user_seq in list(zip(user_group_noisy.index, user_group_noisy.tolist())):
    # Ensure user_id is Python int type
    user_id = int(user_id)
    
    # Truncate to maximum sequence length
    user_seq = user_seq[-max_seq_len:]
    
    # Skip users with sequences too short
    if len(user_seq) < 3:
        continue
        
    # Ensure item_id in user_seq is Python int type
    user_seq = [int(item) for item in user_seq]
        
    # ------ Test sample ------------
    history, seq_len = truncate_or_pad(user_seq[:-1])
    target_data = user_seq[-1]
    label = 1
    
    # Domain handling: following original code logic, domain_id is [0] * max_seq_len
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    test_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Validation sample -------------
    history, seq_len = truncate_or_pad(user_seq[:-2])
    target_data = user_seq[-2]
    label = 1
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    val_noisy.append([user_id, history, target_data, seq_len, label, domain_id, history])
    
    # ------ Training sample -----------
    history, seq_len = truncate_or_pad(user_seq[:-3])
    target_data, _ = truncate_or_pad(user_seq[-seq_len-2:-2])
    label = [1] * seq_len + [PAD] * (max_seq_len - seq_len)
    domain_id = [0] * max_seq_len
    
    # Ensure data types
    history = ensure_python_types(history)
    target_data = ensure_python_types(target_data)
    seq_len = ensure_python_types(seq_len)
    label = ensure_python_types(label)
    domain_id = ensure_python_types(domain_id)
    
    train_noisy.append([user_id, history, target_data, seq_len, label, domain_id])

# Ensure all data are Python native types
train_noisy = ensure_python_types(train_noisy)
val_noisy = ensure_python_types(val_noisy)
test_noisy = ensure_python_types(test_noisy)

# Save sequence data
torch.save(train_noisy, os.path.join(noisy_dir, 'train.pth'))
torch.save(val_noisy, os.path.join(noisy_dir, 'val.pth'))
torch.save(test_noisy, os.path.join(noisy_dir, 'test.pth'))

print(f"  Saved sequence data: training={len(train_noisy)}, validation={len(val_noisy)}, test={len(test_noisy)}")

print("\n" + "="*60)
print("Data processing completed!")
print("="*60)

# 7. Final verification
print(f"\n{'='*40}")
print("Final verification results:")
print(f"{'='*40}")

# Verify data integrity
df_clean = pd.read_csv(csv_path)
df_noisy = pd.read_csv(noisy_csv_path)

print(f"Clean data: {len(df_clean)} rows, {df_clean['user_id'].nunique()} users")
print(f"Noisy data: {len(df_noisy)} rows, {df_noisy['user_id'].nunique()} users")

# Calculate actual noise ratio
actual_noise_ratio = (len(df_noisy) - len(df_clean)) / len(df_clean)
print(f"\n📊 Noise ratio statistics:")
print(f"  Target noise ratio: {NOISE_RATIO*100:.1f}%")
print(f"  Actual noise ratio: {actual_noise_ratio*100:.2f}%")
print(f"  Absolute error: {abs(actual_noise_ratio - NOISE_RATIO)*100:.3f}%")

print(f"\n Output directory structure:")
print(f"{output_path}/")
print(f"└── {dataset_name}/")
print(f"    ├── clean/")
print(f"    │   ├── inter.csv")
print(f"    │   ├── seq2pat_data.pth")
print(f"    │   ├── train.pth")
print(f"    │   ├── val.pth")
print(f"    │   └── test.pth")
print(f"    └── noise_5/")
print(f"        ├── inter.csv")
print(f"        ├── seq2pat_data.pth")
print(f"        ├── train.pth")
print(f"        ├── val.pth")
print(f"        └── test.pth")

print(f"\n Key guarantees:")
print(f"1. All items have interaction count ≥ {item_threshold}")
print(f"2. Precisely controlled noise ratio ≈ 5%")
print(f"3. Random insertion of new interactions without modifying existing ones")
print(f"4. Domain handled according to original code logic (fixed to 0)")
print(f"5. All data are Python native types, avoiding numpy type issues")

print("\n Yelp dataset noise experiment ready!")